In [ ]:
import re

syn_exp = re.compile(r"(.*)__(bottle|box|carton|bag|jar|dispenser|container|can)\..*")
for cat, syn in rows:
    match = syn_exp.fullmatch(syn)
    if match:
        syn = f"{match.group(2)}__of__{match.group(1)}.n.01"
    print(syn)

In [ ]:
cat_exp = re.compile(r"(.*)_(bottle|box|carton|bag|jar|dispenser|container|can)$")
for cat, syn in rows:
    match = cat_exp.fullmatch(cat)
    if match:
        cat = f"{match.group(2)}_of_{match.group(1)}"
    print(cat)

In [ ]:
import csv

with open("D:\BEHAVIOR-1K\asset_pipeline\metadata\custom_synsets.csv", "r") as f:
    data = list(csv.DictReader(f))

In [ ]:
import collections

ctr = collections.Counter(x["synset"] for x in data)
ctr.most_common()

In [ ]:
from nltk.corpus import wordnet as wn


def in_wn(s):
    try:
        wn.synset(s)
        return True
    except:
        return False


def canonicalize(s):
    try:
        return wn.synset(s).name()
    except:
        return s

In [ ]:
# Who has != .n.01 even though their 01 is not in wn?
def fix_syn_no(s):
    parts = s.strip().split(".")
    if parts[-1] == "01":
        return s

    one_below_id = "%.02d" % (int(parts[-1]) - 1)
    one_below = f"{parts[0]}.{parts[1]}.{one_below_id}"
    if in_wn(one_below):
        return s

    # Correct the ID
    for i in range(1, int(parts[-1])):
        candidate_id = "%.02d" % i
        candidate_s = f"{parts[0]}.{parts[1]}.{candidate_id}"
        if not in_wn(candidate_s):
            break
    else:
        raise ValueError("whoa")

    return candidate_s

In [ ]:
for d in data:
    s = canonicalize(fix_syn_no(d["synset"]))
    is_custom = int(not in_wn(s))
    if is_custom:
        hypernyms = (
            ",".join([wn.synset(x).name() for x in d["hypernyms"].split(",")])
            if d["hypernyms"]
            else ""
        )
    else:
        hypernyms = ",".join([x.name() for x in wn.synset(s).hypernyms()])
    is_duplicate = int(ctr[s] > 1)
    print(f'"{s}","{hypernyms}","{is_duplicate}","{is_custom}"')

In [ ]:
with open(r"D:\BEHAVIOR-1K\asset_pipeline\metadata\category_mapping.csv", "r") as f:
    category_mapping = list(csv.DictReader(f))

In [ ]:
for r in category_mapping:
    if r["synset"] not in ctr:
        print(r["synset"])

In [ ]:
import networkx as nx
from nltk.corpus import wordnet as wn
import csv


def get_synset_graph():
    """
    Build the synset graph that includes all wordnet and custom synsets
    returns:
        G: the synset graph
    """
    # Build the legit-synset graph
    G = nx.DiGraph()
    G.add_nodes_from(x.name() for x in wn.all_synsets())
    for parent in wn.all_synsets():
        for child in parent.hyponyms():
            G.add_edge(parent.name(), child.name())

    # Add the illegit-synset (custom) graph
    with open(r"D:\BEHAVIOR-1K\asset_pipeline\metadata\custom_synsets.csv") as f:
        reader = csv.DictReader(f)
        for row in reader:
            child = row["synset"].strip()
            parents = row["hypernyms"].strip().split(",")
            custom = bool(int(row["is_custom"].strip()))
            if not custom or not parents:
                continue
            canonical_parent = wn.synset(parent).name()
            if canonical_parent != parent:
                print(
                    f"Parent {parent} of {child} is not canonical in wordnet, using {canonical_parent} instead"
                )
            if child in G.nodes:
                continue
            assert canonical_parent in G.nodes, f"Could not find {canonical_parent}"
            assert child not in G.nodes, f"Custom synset {child} already in wordnet!"
            G.add_edge(canonical_parent, child)
    return G


def canonicalize(s):
    try:
        return wn.synset(s).name()
    except:
        return s


def wn_synset_exists(synset):
    try:
        wn.synset(synset)
        return True
    except:
        return False

In [ ]:
G = get_synset_graph()